In [ ]:
# Mount Google Drive and set up repo
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os
REPO_PATH = '/content/lung-nodule-fusion-xai'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/lung-nodule-fusion-xai.git', REPO_PATH], check=True)
os.chdir(REPO_PATH)


In [ ]:
# Install required packages (skips torch — Colab has it pre-installed)
!pip install -q pylidc==0.2.2 SimpleITK==2.3.1 nibabel==5.2.1 pyradiomics==3.1.0 \
               scikit-learn==1.5.0 xgboost==2.0.3 pandas==2.2.2 pyarrow==16.1.0


In [ ]:
# Configure pylidc to find LIDC-IDRI data
# LIDC-IDRI should be downloaded to /content/drive/MyDrive/LIDC-IDRI/
import pylidc as pl
import configparser, os

cfg = configparser.ConfigParser()
cfg['pylidc'] = {'dicom_path': '/content/drive/MyDrive/LIDC-IDRI'}
with open(os.path.expanduser('~/.pylidcrc'), 'w') as f:
    cfg.write(f)
print("pylidc configured")


In [ ]:
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')

from src.data_loading.lidc_loader import load_and_split

df = load_and_split(
    lidc_path='/content/drive/MyDrive/LIDC-IDRI',
    interim_path='/content/drive/MyDrive/lung_nodule_interim',
    processed_path='/content/drive/MyDrive/lung_nodule_processed',
    n_folds=5,
    seed=42,
)
print(f"Total nodules: {len(df)}")
print(f"Malignant: {(df.label==1).sum()}, Benign: {(df.label==0).sum()}")
print(df.head())


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fold_counts = df.groupby(['fold', 'label']).size().unstack(fill_value=0)
fold_counts.plot(kind='bar', figsize=(8, 4), title='Fold Distribution by Label')
plt.xlabel('Fold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/lung_nodule_processed/fold_distribution.png', dpi=150)
plt.show()
print("Fold distribution saved")
